# Simulation: drogued drifter and point particle runs

Run drogued drifter and point particle simulations from observed
deployment positions. Six drifters (298--303) with 3 m drogues are
simulated in effective currents (Eulerian + Stokes) using the pipeline
from notebook 08. Surface and 3 m point particles provide baselines.

This notebook produces output files only -- no visualization.

## Imports

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode
from parcels.kernels import AdvectionEE

from drogued_drifters.drifter import DroguedDrifter, make_dd_velocity_interpolator

## Parameters

In [ ]:
LON_MIN = 9.0
LON_MAX = 13.0
LAT_MIN = 53.5
LAT_MAX = 56.0
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
DROGUE_DEPTH = 3.0
DT = 300.0
RUNTIME_HOURS = 288
CSV_PATH = "data/drifters_clean.csv"
OUTPUT_DIR = "output"
CMEMS_DIR = "data/cmems"

## Load observed trajectories

In [3]:
df = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])
drifter_ids = sorted(df["D_number"].unique())

# Extract deployment position and time (first record per drifter)
deployments = {}
for d_num in drifter_ids:
    first = df[df["D_number"] == d_num].iloc[0]
    deployments[d_num] = {
        "lon": first["Longitude"],
        "lat": first["Latitude"],
        "time": first["date_UTC"],
    }

for d_num, dep in deployments.items():
    print(f"  {d_num}: ({dep['lat']:.4f}N, {dep['lon']:.4f}E) at {dep['time']}")

  298: (54.5117N, 10.2004E) at 2023-04-24 08:28:52
  299: (54.5124N, 10.2006E) at 2023-04-24 08:56:12
  300: (54.5093N, 10.1968E) at 2023-04-24 07:57:27
  301: (54.5101N, 10.1968E) at 2023-04-24 08:02:51
  302: (54.5095N, 10.1965E) at 2023-04-24 07:59:57
  303: (54.5096N, 10.1968E) at 2023-04-24 08:00:45


## Build effective current field

Same pipeline as notebook 08: load CMEMS physics and wave partitions,
compute depth-dependent Stokes drift from wave partitions, add to
Eulerian currents, add z=0 layer, build FieldSet.

In [ ]:
LON = slice(LON_MIN, LON_MAX)
LAT = slice(LAT_MIN, LAT_MAX)
TIME = slice(TIME_START, TIME_END)

ds_phy = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_phy_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
).load()

ds_phy

In [ ]:
WAVE_VARS = [
    "VHM0_WW", "VTM01_WW", "VMDR_WW",
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",
]

ds_wav = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_wav_anfc_PT1H-i.nc").sel(
    longitude=LON, latitude=LAT, time=TIME,
)[WAVE_VARS].load()

ds_wav

### Stokes drift profiles from wave partitions

Deep-water monochromatic approximation per partition, evaluated at each
Eulerian depth level including z=0.

In [6]:
g = 9.81
depth_levels = np.concatenate([[0.0], ds_phy.depth.values])

PARTITIONS = [
    ("VHM0_WW", "VTM01_WW", "VMDR_WW"),
    ("VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
    ("VHM0_SW2", "VTM01_SW2", "VMDR_SW2"),
]

u_stokes = xr.DataArray(
    np.zeros((len(ds_wav.time), len(depth_levels), len(ds_wav.latitude), len(ds_wav.longitude))),
    dims=["time", "depth", "latitude", "longitude"],
    coords={"time": ds_wav.time, "depth": depth_levels,
            "latitude": ds_wav.latitude, "longitude": ds_wav.longitude},
)
v_stokes = u_stokes.copy()

for hs_var, tp_var, dir_var in PARTITIONS:
    Hs = ds_wav[hs_var]
    T = ds_wav[tp_var]
    dir_from = ds_wav[dir_var]
    valid = Hs.notnull() & T.notnull() & dir_from.notnull() & (T > 0)
    A = Hs / 2.0
    sigma = 2 * np.pi / T
    k = sigma**2 / g
    theta = np.deg2rad(270.0 - dir_from)
    stokes_surf = A**2 * sigma * k

    for i, z in enumerate(depth_levels):
        decay = np.exp(-2 * k * float(z))
        u_stokes[dict(depth=i)] += (stokes_surf * decay * np.cos(theta)).where(valid, 0.0).fillna(0.0)
        v_stokes[dict(depth=i)] += (stokes_surf * decay * np.sin(theta)).where(valid, 0.0).fillna(0.0)

### Interpolate Stokes onto physics grid and build effective current

In [7]:
ds_stokes = xr.Dataset({"u_stokes": u_stokes, "v_stokes": v_stokes})

ds_stokes_phys = ds_stokes.interp(
    longitude=ds_phy.longitude,
    latitude=ds_phy.latitude,
    method="linear",
).fillna(0.0)

# Add z=0 layer to Eulerian by copying shallowest level
ds_phy_z0 = ds_phy.isel(depth=0).assign_coords(depth=0.0)
ds_phy_ext = xr.concat([ds_phy_z0, ds_phy], dim="depth")

# Apply land mask from Eulerian data
land_mask = ds_phy_ext["uo"].isnull()
ds_stokes_phys["u_stokes"] = ds_stokes_phys["u_stokes"].where(~land_mask)
ds_stokes_phys["v_stokes"] = ds_stokes_phys["v_stokes"].where(~land_mask)

# Effective current = Eulerian + Stokes
ds_eff = xr.Dataset({
    "U": (ds_phy_ext["uo"] + ds_stokes_phys["u_stokes"]).fillna(0.0),
    "V": (ds_phy_ext["vo"] + ds_stokes_phys["v_stokes"]).fillna(0.0),
})
ds_eff

<xarray.Dataset> Size: 840MB
Dimensions:    (depth: 6, latitude: 150, longitude: 143, time: 408)
Coordinates:
  * depth      (depth) float64 48B 0.0 0.5016 1.516 2.548 3.602 4.684
  * latitude   (latitude) float32 600B 53.51 53.52 53.54 ... 55.96 55.97 55.99
  * longitude  (longitude) float32 572B 9.042 9.069 9.097 ... 12.93 12.96 12.99
  * time       (time) datetime64[ns] 3kB 2023-04-24 ... 2023-05-10T23:00:00
Data variables:
    U          (time, latitude, longitude, depth) float64 420MB 0.0 0.0 ... 0.0
    V          (time, latitude, longitude, depth) float64 420MB 0.0 0.0 ... 0.0

### Build FieldSet

In [ ]:
ds_eff["grid"] = xr.DataArray(
    data=0,
    attrs={
        "cf_role": "grid_topology",
        "topology_dimension": 2,
        "node_dimensions": "longitude latitude",
        "face_dimensions": (
            "longitude:longitude (padding: none) "
            "latitude:latitude (padding: none)"
        ),
        "vertical_dimensions": "depth:depth (padding: none)",
        "node_coordinates": "longitude latitude",
    },
)

fieldset = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")

## Drogued drifter interpolator

Replace the default Parcels velocity interpolator with `make_dd_velocity_interpolator`.
At each RHS evaluation, the interpolator extracts the full velocity profile at each
particle position (bilinear in t, y, x at every depth level), builds a fast
`sample_uv(z)` closure, and runs `DroguedDrifter.get_final_drift_batch` to obtain the
steady-state drift velocity. `AdvectionEE` then steps particles with that drift.

In [ ]:
dd = DroguedDrifter()
fieldset.UV.vector_interp_method = make_dd_velocity_interpolator(dd, spherical=True)

# Point particle runs need a separate FieldSet with the default interpolator.
fieldset_pp = FieldSet.from_sgrid_conventions(ds_eff, mesh="spherical")


def DeleteOOB(particles, fieldset):
    state = np.asarray(particles.state)
    oob = (state == StatusCode.ErrorOutOfBounds) | (state == StatusCode.ErrorThroughSurface)
    if np.any(oob):
        particles.state = np.where(oob, StatusCode.Delete, state)

## Run drogued drifter simulation

Deploy 6 virtual drifters at the observed deployment positions and
times.

In [ ]:
RUNTIME = RUNTIME_HOURS * 3600
SHALLOWEST_DEPTH = float(ds_eff.depth.values[0])
OUTPUTDT = 3600.0
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

release_lons = [deployments[did]["lon"] for did in drifter_ids]
release_lats = [deployments[did]["lat"] for did in drifter_ids]
release_times = [np.datetime64(deployments[did]["time"]) for did in drifter_ids]

dd_store = str(output_dir / "sim_drogued_drifter.zarr")
shutil.rmtree(dd_store, ignore_errors=True)

pset_drifter = ParticleSet(
    fieldset=fieldset,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[SHALLOWEST_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_drifter.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=dd_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

In [ ]:
# Convert zarr to CSV for downstream notebooks.
# Zarr trajectory IDs are particle indices (0 = drifter_ids[0], etc.);
# some may be absent if the particle was deleted before any output was written.
ds_dd = xr.open_zarr(dd_store)

# trajectory coord holds the integer particle indices that survived
traj_ids = ds_dd.trajectory.values  # e.g. [0, 3, 4] if others were deleted
id_to_did = {i: drifter_ids[i] for i in range(len(drifter_ids))}

rows = []
for j, tid in enumerate(traj_ids):
    did = id_to_did.get(int(tid))
    if did is None:
        continue
    lons = ds_dd.lon.values[j, :]
    lats = ds_dd.lat.values[j, :]
    times = ds_dd.time.values[j, :]
    valid = np.isfinite(lons)
    for lon, lat, t in zip(lons[valid], lats[valid], times[valid]):
        rows.append({"D_number": did, "time": t, "lon": lon, "lat": lat})

df_drogued = pd.DataFrame(rows)
df_drogued.to_csv(output_dir / "sim_drogued_drifter.csv", index=False)
print(f"Saved {len(df_drogued)} rows to {output_dir / 'sim_drogued_drifter.csv'}")

## Run surface point particles

Surface (z=0) point particles with AdvectionEE, same deployment positions and times.

In [ ]:
surface_store = str(output_dir / "sim_surface.zarr")
shutil.rmtree(surface_store, ignore_errors=True)

pset_surface = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[SHALLOWEST_DEPTH] * len(drifter_ids),
    time=release_times,
)
pset_surface.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=surface_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

## Run 3 m point particles

Point particles at drogue depth with AdvectionEE, same deployment positions and times.

In [ ]:
DROGUE_DEPTH_LEVEL = float(ds_eff.depth.sel(depth=DROGUE_DEPTH, method="nearest"))

drogue_store = str(output_dir / "sim_3m.zarr")
shutil.rmtree(drogue_store, ignore_errors=True)

pset_drogue = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons,
    lat=release_lats,
    z=[DROGUE_DEPTH_LEVEL] * len(drifter_ids),
    time=release_times,
)
pset_drogue.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=drogue_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

## Save observed trajectories for convenience

Copy the clean observed CSV into the output directory so downstream
notebooks can find everything in one place.

In [14]:
obs_dst = output_dir / "obs_science.csv"
shutil.copy2(CSV_PATH, obs_dst)
print(f"Copied observed data to {obs_dst}")

Copied observed data to output/obs_science.csv


## Summary

In [ ]:
# Drogued drifter summary
n_drifters = df_drogued["D_number"].nunique()
print(f"Drogued drifter: {n_drifters} drifters, {len(df_drogued)} total records")
for did in drifter_ids:
    sub = df_drogued[df_drogued["D_number"] == did]
    if len(sub) > 0:
        print(f"  D{did}: {len(sub)} steps, final ({sub.iloc[-1]['lat']:.4f}N, {sub.iloc[-1]['lon']:.4f}E)")

# Surface point particle summary
ds_s = xr.open_zarr(surface_store)
n_surf = ds_s.sizes["trajectory"]
print(f"\nSurface point particles: {n_surf} particles")
for i in range(n_surf):
    last_valid = np.where(np.isfinite(ds_s.lon.values[i, :]))[0]
    if len(last_valid) > 0:
        idx = last_valid[-1]
        print(f"  Particle {i}: {len(last_valid)} obs, final ({ds_s.lat.values[i, idx]:.4f}N, {ds_s.lon.values[i, idx]:.4f}E)")

# 3 m point particle summary
ds_d = xr.open_zarr(drogue_store)
n_drg = ds_d.sizes["trajectory"]
print(f"\n3 m point particles: {n_drg} particles")
for i in range(n_drg):
    last_valid = np.where(np.isfinite(ds_d.lon.values[i, :]))[0]
    if len(last_valid) > 0:
        idx = last_valid[-1]
        print(f"  Particle {i}: {len(last_valid)} obs, final ({ds_d.lat.values[i, idx]:.4f}N, {ds_d.lon.values[i, idx]:.4f}E)")

print(f"\nOutput files:")
print(f"  {dd_store}")
print(f"  {output_dir / 'sim_drogued_drifter.csv'}")
print(f"  {surface_store}")
print(f"  {drogue_store}")
print(f"  {output_dir / 'obs_science.csv'}")